# CEIT Corrosion Monitoring Workflow

This notebook walks through the `CeitCorrosionMonitoring` workflow as *explicit, auditable notebook steps* instead of hiding the SHM lookup, POST, and GET calls behind a helper function.

**What this notebook covers**

- **Import** the required SDK components from the local multi-repo workspace.
- **Configure** the JSON file path, API root, CEIT metadata, and token source.
- **Resolve** each CEIT `sensor_identifier` to exactly one SHM signal through `SignalHistory`.
- **Prepare** validated CEIT measurements and derive one stable result series per sensor and metric.
- **Upload** the results into one shared backend analysis named `CeitCorrosionMonitoring`.
- **Retrieve** persisted rows back from the API and reconstruct the normalized CEIT frame.
- **Plot** the retrieved data through the CEIT pyecharts dropdown plot.

**How to use it**

- Run the notebook *top to bottom* the first time.
- Inspect the Mermaid diagrams before each stage to understand *inputs, transformations, and outputs*.
- If a sensor identifier does not resolve uniquely to one SHM signal, the notebook will stop before uploading any result rows.

## Mermaid Color Legend

The workflow diagrams in this notebook use a consistent visual language.

- <span style="color:#2B6F77;"><strong>Blue nodes</strong></span>: API calls, services, or external system interactions.
- <span style="color:#5E8C61;"><strong>Green nodes</strong></span>: validated outputs, identifiers, or reconstructed result frames.
- <span style="color:#C08B3E;"><strong>Amber nodes</strong></span>: transformation, serialization, or intermediate processing steps.
- <span style="color:#C47A2C;"><strong>Orange decision/request nodes</strong></span>: branching logic, runtime choices, or fail-fast validation.
- <span style="color:#365A73;"><strong>Blue-grey connectors</strong></span>: data flow between notebook steps.

*Read the diagrams from top to bottom to follow the lifecycle from raw CEIT JSON measurements to plotted backend results.*

## Imports

This section bootstraps the notebook so it can import the core SDK, the results SDK, and the SHM SDK directly from the current multi-repo workspace.

**Why this is explicit**

- The notebook is meant to be runnable from the repository checkout without relying on a separately installed editable environment.
- Import resolution is made visible up front so there is no hidden state about where `owi.metadatabase.*` modules come from.
- The imported APIs stay close to the 1.0 notebook style, with the only extra piece being the SHM client required for `SignalHistory` resolution.

In [14]:
!uv pip install "owi-metadatabase[shm]"


Using Python 3.13.12 environment at: /home/pietro.dantuono@24SEA.local/Projects/SMARTLIFE/OWI-metadatabase-SDK/owi-metadatabase-results-sdk/.venv
Audited 1 package in 4ms


In [15]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from owi.metadatabase._utils.utils import load_token_from_env_file
from owi.metadatabase.geometry.io import GeometryAPI
from owi.metadatabase.locations.io import LocationsAPI
from owi.metadatabase.shm import ShmAPI

from owi.metadatabase.results import CeitResultsService, ResultsAPI, load_ceit_measurements
from owi.metadatabase.results.analyses import CEIT_METRICS, ceit_frame_from_measurements


## Constants

This section defines the *runtime configuration* for the notebook.

**Review these values before execution**

- **`DATA_FILE`**: source JSON file containing the raw CEIT measurements.
- **`BASE_URL`**: API root used for locations, geometry, results, and SHM calls.
- **`PROJECTSITE`**, **`ASSETLOCATION`**, and **`MODEL_DEFINITION`**: metadata context used to resolve the correct backend objects.
- **`ANALYSIS_NAME`**: the shared analysis reused or created by this notebook.
- **`TOKEN`**: authentication token loaded from the repository environment file.

*Outcome:* the notebook has one explicit place where environment-specific values can be checked or changed before any API request is made.*

In [4]:
WORKSPACE_ROOT = Path().resolve().parent
EXPECTED_DATA_FILE = WORKSPACE_ROOT / "scripts" / "data" / "owiMeasFile_24sea.json"
LEGACY_DATA_FILE = WORKSPACE_ROOT / "scripts" / "data" / "MeasFile_24sea.json"
DEFAULT_ENV_FILE = WORKSPACE_ROOT / ".env"
TOKEN_ENV_VAR = "OWI_METADATABASE_API_TOKEN"
DEFAULT_RESULTS_API_ROOT = "https://owimetadatabase-dev.azurewebsites.net/api/v1"

BASE_URL = DEFAULT_RESULTS_API_ROOT
DATA_FILE = EXPECTED_DATA_FILE if EXPECTED_DATA_FILE.exists() else LEGACY_DATA_FILE
PROJECTSITE = "Willow"
ASSETLOCATION = "CEIT"
MODEL_DEFINITION = "CEIT Willow"
ANALYSIS_NAME = "CeitCorrosionMonitoring"
TOKEN = load_token_from_env_file(DEFAULT_ENV_FILE, TOKEN_ENV_VAR)

display(pd.DataFrame([{"data_file": str(DATA_FILE),
                       "api_root": BASE_URL,
                "projectsite": PROJECTSITE,
                "assetlocation": ASSETLOCATION,
                "model_definition": MODEL_DEFINITION,
                "analysis_name": ANALYSIS_NAME,
                "token_available": TOKEN is not None,
            }
        ]
    ).T.rename(columns={0: "value"})
)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"No CEIT data file was found. Checked {EXPECTED_DATA_FILE} and {LEGACY_DATA_FILE}."
    )

if TOKEN is None:
    raise ValueError(
        f"Set {TOKEN_ENV_VAR} in {DEFAULT_ENV_FILE} or export it before running this notebook."
    )


,value
data_file,/home/pietro.dantuono@24SEA.local/Projects/SMA...
api_root,https://owimetadatabase-dev.azurewebsites.net/...
projectsite,Willow
assetlocation,CEIT
model_definition,CEIT Willow
analysis_name,CeitCorrosionMonitoring
token_available,True


## Resolve CEIT Metadata

Before looking at the measurements, the notebook resolves the backend metadata context for the shared analysis.

**What is resolved here**

- The CEIT project site identifier.
- The unique CEIT asset location used to persist location-scoped result rows.
- The CEIT Willow model definition identifier.
- Any pre-existing `CeitCorrosionMonitoring` analysis row already present in the results backend.

*Outcome:* later upload and retrieval cells can work with explicit ids instead of repeating title-based lookups.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["BASE_URL + TOKEN"] --> B["locations_api = LocationsAPI(...)"]
    A --> C["geometry_api = GeometryAPI(...)"]
    A --> D["results_api = ResultsAPI(...)"]
    B --> E["site_detail = locations_api.get_projectsite_detail(projectsite=PROJECTSITE)"]
    E --> F["site_id"]
    B --> G["assetlocations = locations_api.get_assetlocations(projectsite=PROJECTSITE)['data']"]
    G --> H["matching_locations"]
    H --> I["location_id"]
    C --> J["model_definition_id = geometry_api.get_modeldefinition_id(projectsite=PROJECTSITE, model_definition=MODEL_DEFINITION)['id']"]
    D --> K["existing_analysis_rows = results_api.list_analyses(name=ANALYSIS_NAME)['data']"]
    K --> L["existing_analysis_id"]
    D --> M["service = CeitResultsService(...)"]
    I --> M
    J --> M

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class B,C,D,E,G,K,M api;
    class A,H transform;
    class F,I,J,L output;
```


In [5]:
locations_api = LocationsAPI(api_root=BASE_URL, token=TOKEN)
geometry_api = GeometryAPI(api_root=BASE_URL, token=TOKEN)
results_api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
shm_api = ShmAPI(api_root=BASE_URL, token=TOKEN)

site_detail = locations_api.get_projectsite_detail(projectsite=PROJECTSITE)
site_id = int(site_detail["id"])

assetlocations = locations_api.get_assetlocations(projectsite=PROJECTSITE)["data"]
if assetlocations.empty:
    raise ValueError(f"No asset locations were found for project site {PROJECTSITE}.")

matching_locations = assetlocations.loc[
    assetlocations["title"].astype(str).str.casefold() == ASSETLOCATION.casefold()
].copy()
if matching_locations.empty:
    raise ValueError(
        f"No asset location titled {ASSETLOCATION!r} was found under project site {PROJECTSITE!r}."
    )
if len(matching_locations) > 1:
    display(matching_locations)
    raise ValueError(
        f"Expected one asset location titled {ASSETLOCATION!r} but found {len(matching_locations)} rows."
    )

location_id = int(matching_locations.iloc[0]["id"])
model_definition_id = int(
    geometry_api.get_modeldefinition_id(
        projectsite=PROJECTSITE,
        model_definition=MODEL_DEFINITION,
    )["id"]
)
existing_analysis_rows = results_api.list_analyses(name=ANALYSIS_NAME)["data"]
if len(existing_analysis_rows) > 1:
    display(existing_analysis_rows)
    raise ValueError(
        f"Expected at most one analysis named {ANALYSIS_NAME!r} but found {len(existing_analysis_rows)} rows."
    )
existing_analysis_id = None if existing_analysis_rows.empty else int(existing_analysis_rows.iloc[0]["id"])
service = CeitResultsService(
    api=results_api,
    model_definition_id=model_definition_id,
    location_id=location_id,
 )

metadata_summary = pd.DataFrame(
    [
        {
            "site_id": site_id,
            "location_id": location_id,
            "model_definition_id": model_definition_id,
            "existing_analysis_id": existing_analysis_id,
        }
    ]
).T.rename(columns={0: "value"})

display(metadata_summary)
display(
    matching_locations.loc[:, [
        column
        for column in ["id", "title", "northing", "easting"]
        if column in matching_locations.columns
    ]]
)


,value
site_id,77
location_id,3443
model_definition_id,18
existing_analysis_id,58


,id,title,northing,easting
0,3443,CEIT,0.0,0.0


## Load and Normalize the CEIT Measurements

This stage validates the JSON file through the CEIT analysis helpers and expands each raw measurement into one long-form row per metric.

**What happens here**

- The raw JSON file is parsed through `load_ceit_measurements(...)`.
- Each measurement row becomes five normalized observations through `ceit_frame_from_measurements(...)`.
- The notebook derives the unique sensor identifiers that must be matched against SHM `SignalHistory`.

*Outcome:* the raw CEIT export is converted into a deterministic input frame for signal resolution, upload, and round-trip validation.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["DATA_FILE"] --> B["measurements = load_ceit_measurements(DATA_FILE)"]
    B --> C["measurement_frame = ceit_frame_from_measurements(measurements)"]
    B --> D["unique_sensors"]
    B --> E["expected_observation_count"]

    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,B transform;
    class C,D,E output;
```

In [6]:
measurements = load_ceit_measurements(DATA_FILE)
measurement_frame = ceit_frame_from_measurements(measurements)
unique_sensors = sorted({measurement.sensor_identifier for measurement in measurements})
expected_observation_count = len(measurements) * len(CEIT_METRICS)

measurement_summary = pd.DataFrame(
    [
        {
            "raw_measurement_rows": len(measurements),
            "unique_sensors": len(unique_sensors),
            "normalized_observations": len(measurement_frame),
            "expected_observations": expected_observation_count,
        }
    ]
).T.rename(columns={0: "value"})

display(measurement_summary)
display(measurement_frame.head())
display(
    measurement_frame.groupby(["sensor_identifier", "metric"]).size().rename("points").reset_index()
        .sort_values(["sensor_identifier", "metric"])
        .reset_index(drop=True)
)


,value
raw_measurement_rows,8
unique_sensors,3
normalized_observations,40
expected_observations,40


,sensor_identifier,timestamp,metric,value
0,DA8F,2026-03-12T12:05:12+00:00,temperature,-40.84765
1,DA8F,2026-03-12T12:05:12+00:00,battery,4.13714
2,DA8F,2026-03-12T12:05:12+00:00,tof,1653.43945
3,DA8F,2026-03-12T12:05:12+00:00,amplitude,43.00000
4,DA8F,2026-03-12T12:05:12+00:00,meas_gain,1.39999


,sensor_identifier,metric,points
0,DA87,amplitude,2
1,DA87,battery,2
2,DA87,meas_gain,2
3,DA87,temperature,2
4,DA87,tof,2
5,DA8F,amplitude,4
6,DA8F,battery,4
7,DA8F,meas_gain,4
8,DA8F,temperature,4
9,DA8F,tof,4


## Resolve SHM Signals from `SignalHistory`

This stage maps each unique CEIT `sensor_identifier` to exactly one SHM signal.

**Resolution rules**

- Try an exact `legacy_signal_id` match first because the CEIT identifiers are string-like.
- Try `sensor_serial_number` only when the identifier is numeric.
- De-duplicate to unique `signal_id` values and prefer rows with a better status or later timestamp.
- Stop the notebook before upload if any CEIT sensor maps to zero or multiple SHM signals.

*Outcome:* upload can attach each persisted result row to the correct SHM signal with a reproducible audit trail.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["shm_api.list_sensors()['data']"] --> B["sensor_frame"]
    C["shm_api.list_signal_history()['data']"] --> D["signal_history_frame"]
    E["unique_sensors"] --> F["for sensor_identifier in unique_sensors"]
    B --> G["_resolve_signal_history(sensor_identifier)"]
    D --> G
    F --> G
    G --> H["candidates, resolution_state"]
    H --> I["selected_signal_id"]
    I --> J{"len(candidates) == 1 and selected_signal_id is not None?"}
    J -- yes --> K["signal_ids_by_sensor + signal_history_by_sensor"]
    J -- no --> L["resolution_candidates"]
    H --> M["resolution_rows"]
    M --> N["resolution_frame"]
    L --> O{"resolution_candidates?"}
    N --> O
    O -- yes --> P["display(candidate_frame) + raise ValueError"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;
    classDef decision fill:#FDEBD3,stroke:#C47A2C,color:#5A3412;

    class A,C api;
    class F,G,H,I,M transform;
    class B,D,E,K,L,N output;
    class J,O,P decision;
```

In [7]:
from collections.abc import Mapping
from typing import Any

sensor_frame = shm_api.list_sensors()["data"]
signal_history_frame = shm_api.list_signal_history()["data"]

def _json_ready_value(value: Any) -> Any:
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if value is pd.NaT:
        return None
    if isinstance(value, Mapping):
        return {str(key): _json_ready_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_json_ready_value(item) for item in value]
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if hasattr(value, "item") and not isinstance(value, (str, bytes, bytearray)):
        try:
            return value.item()
        except (TypeError, ValueError):
            return str(value)
    return value

def _json_ready_mapping(mapping: Mapping[Any, Any]) -> dict[str, Any]:
    return {str(key): _json_ready_value(value) for key, value in mapping.items()}

def _rank_signal_history_rows(candidates: pd.DataFrame) -> pd.DataFrame:
    ranked = candidates.copy()
    if "activity_start_timestamp" in ranked.columns:
        ranked["sort_timestamp"] = pd.to_datetime(
            ranked["activity_start_timestamp"], errors="coerce", utc=True
        )
    elif "timestamp" in ranked.columns:
        ranked["sort_timestamp"] = pd.to_datetime(
            ranked["timestamp"], errors="coerce", utc=True
        )
    else:
        ranked["sort_timestamp"] = pd.NaT

    if "status" in ranked.columns:
        ranked["status_priority"] = (
            ranked["status"].astype(str).str.lower().map({"ok": 0, "active": 1, "installed": 2}).fillna(9)
        )
    else:
        ranked["status_priority"] = 9

    ranked = ranked.sort_values(
        ["status_priority", "sort_timestamp"],
        ascending=[True, False],
        na_position="last",
    )
    return ranked.drop_duplicates(subset=["signal_id"], keep="first").reset_index(drop=True)

def _resolve_signal_history(sensor_identifier: str) -> tuple[pd.DataFrame, str]:
    exact_history = pd.DataFrame()
    if "legacy_signal_id" in signal_history_frame.columns:
        exact_history = signal_history_frame.loc[
            signal_history_frame["legacy_signal_id"].astype(str).str.casefold() == sensor_identifier.casefold()
        ].copy()
    if not exact_history.empty:
        ranked_exact_history = _rank_signal_history_rows(exact_history)
        return (
            ranked_exact_history,
            "resolved" if len(ranked_exact_history) == 1 else "ambiguous_history",
        )

    sensor_candidates = sensor_frame.loc[
        sensor_frame["serial_number"].astype(str).str.startswith(f"{sensor_identifier}-", na=False)
        | (sensor_frame["serial_number"].astype(str).str.casefold() == sensor_identifier.casefold())
    ].copy()
    if sensor_candidates.empty:
        return pd.DataFrame(), "no_sensor"
    if len(sensor_candidates) > 1:
        return sensor_candidates.reset_index(drop=True), "ambiguous_sensor"

    sensor_row = sensor_candidates.iloc[0]
    sensor_id = int(sensor_row["id"])
    history_candidates = signal_history_frame.loc[
        signal_history_frame["sensor_serial_number"] == sensor_id
    ].copy()
    if history_candidates.empty:
        return pd.DataFrame(), "no_signal_history"

    history_candidates["sensor_record_id"] = sensor_id
    history_candidates["sensor_serial_number_text"] = str(sensor_row["serial_number"])
    ranked_history = _rank_signal_history_rows(history_candidates)
    return ranked_history, "resolved" if len(ranked_history) == 1 else "ambiguous_history"

signal_ids_by_sensor: dict[str, int] = {}
signal_history_by_sensor: dict[str, dict[str, Any]] = {}
resolution_rows: list[dict[str, Any]] = []
resolution_candidates: dict[str, pd.DataFrame] = {}

for sensor_identifier in unique_sensors:
    candidates, resolution_state = _resolve_signal_history(sensor_identifier)
    signal_ids = [] if candidates.empty or "signal_id" not in candidates.columns else [
        int(value) for value in candidates["signal_id"].dropna().tolist()
    ]
    selected_signal_id = signal_ids[0] if len(signal_ids) == 1 else None
    selected_strategy = None
    selected_sensor_serial = None
    if len(candidates) == 1:
        if "legacy_signal_id" in candidates.columns and pd.notna(candidates.iloc[0].get("legacy_signal_id")):
            selected_strategy = "legacy_signal_id"
        elif "sensor_serial_number_text" in candidates.columns:
            selected_strategy = "sensor_serial_number"
            selected_sensor_serial = str(candidates.iloc[0]["sensor_serial_number_text"])

    resolution_rows.append(
        {
            "sensor_identifier": sensor_identifier,
            "resolution_state": resolution_state,
            "candidate_signal_count": len(signal_ids),
            "candidate_signal_ids": ", ".join(str(value) for value in signal_ids),
            "selected_signal_id": selected_signal_id,
            "match_strategy": selected_strategy,
            "sensor_serial_number": selected_sensor_serial,
        }
    )

    if len(candidates) == 1 and selected_signal_id is not None:
        signal_ids_by_sensor[sensor_identifier] = selected_signal_id
        signal_history_by_sensor[sensor_identifier] = _json_ready_mapping(
            candidates.iloc[0].to_dict()
        )
    else:
        resolution_candidates[sensor_identifier] = candidates

resolution_frame = pd.DataFrame(resolution_rows).sort_values("sensor_identifier").reset_index(drop=True)
display(resolution_frame)

if resolution_candidates:
    for sensor_identifier, candidate_frame in resolution_candidates.items():
        print(f"SignalHistory candidates for {sensor_identifier}:")
        display(candidate_frame)
    raise ValueError(
        "Not all CEIT sensor identifiers resolved to exactly one SHM signal. Inspect the candidate frames above."
    )


,sensor_identifier,resolution_state,candidate_signal_count,candidate_signal_ids,selected_signal_id,match_strategy,sensor_serial_number
0,DA87,resolved,1,371,371,sensor_serial_number,DA87-06
1,DA8F,resolved,1,369,369,sensor_serial_number,DA8F-01
2,DA9D,resolved,1,370,370,sensor_serial_number,DA9D-05


## Build and Upload the Shared Analysis

This section turns the resolved CEIT measurements into backend payloads and writes them into one shared analysis named `CeitCorrosionMonitoring`.

**What happens here**

- Build the shared `AnalysisDefinition` used for the upload.
- Preview the serialized result payloads that will be created per sensor and metric.
- Reuse the existing shared analysis if it already exists, otherwise create it.
- Upsert each stable sensor-metric series so reruns are idempotent and append only new timestamps.

*Outcome:* one shared analysis stores all CEIT sensors while each result row stays linked to the correct SHM signal.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["ANALYSIS_NAME + DATA_FILE"] --> B["analysis_definition = service.build_shared_analysis_definition(...)"]
    B --> C["analysis_payload"]
    D["measurements_by_sensor + signal_ids_by_sensor + signal_history_by_sensor + additional_data_by_sensor"] --> E["preview_series = service.build_sensor_results(...)"]
    E --> F["result_payload_preview"]
    G["DATA_FILE + site_id + location_id + ANALYSIS_NAME"] --> H["upload_result = service.upsert_measurements_to_shared_analysis(...)"]
    D --> H
    H --> I["analysis_id + analysis_created"]
    H --> J["upload_summary"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class B,E,H api;
    class A,D,F,G transform;
    class C,I,J output;
```

In [8]:
from rich import print
analysis_definition = service.build_shared_analysis_definition(
    analysis_name=ANALYSIS_NAME,
    source=str(DATA_FILE),
)
analysis_payload = service.analysis_serializer.to_payload(analysis_definition)
print(analysis_payload)


{
    'name': 'CeitCorrosionMonitoring',
    'model_definition_id': 18,
    'location_id': 3443,
    'source_type': 'ceit-json',
    'source': 
'/home/pietro.dantuono@24SEA.local/Projects/SMARTLIFE/OWI-metadatabase-SDK/owi-metadatabase-results-sdk/scripts/dat
a/MeasFile_24sea.json',
    'description': 'Shared CEIT corrosion monitoring analysis across sensors.',
    'additional_data': {'analysis_family': 'CEITSensor', 'shared_analysis': True}
}

In [9]:
analysis_definition = service.build_shared_analysis_definition(
    analysis_name=ANALYSIS_NAME,
    source=str(DATA_FILE),
)
analysis_payload = service.analysis_serializer.to_payload(analysis_definition)

measurements_by_sensor = {
    sensor_identifier: [
        measurement
        for measurement in measurements
        if measurement.sensor_identifier == sensor_identifier
    ]
    for sensor_identifier in unique_sensors
}
additional_data_by_sensor = {
    sensor_identifier: {
        "input_file": DATA_FILE.name,
        "match_strategy": signal_history_by_sensor[sensor_identifier].get("match_strategy"),
    }
    for sensor_identifier in unique_sensors
}

preview_series = []
for sensor_identifier in unique_sensors:
    preview_series.extend(
        service.build_sensor_results(
            sensor_identifier,
            measurements_by_sensor[sensor_identifier],
            site_id=site_id,
            location_id=location_id,
            analysis_name=ANALYSIS_NAME,
            use_stable_series_keys=True,
            signal_id=signal_ids_by_sensor[sensor_identifier],
            signal_history=signal_history_by_sensor[sensor_identifier],
            additional_data=additional_data_by_sensor[sensor_identifier],
        )
    )
result_payload_preview = [
    service.result_serializer.to_payload(series, analysis_id=0)
    for series in preview_series
 ]

display(pd.DataFrame([analysis_payload]))
display(pd.DataFrame(result_payload_preview[:5]))

# Add progressbar
upload_result = service.upsert_measurements_to_shared_analysis(
    DATA_FILE,
    site_id=site_id,
    location_id=location_id,
    analysis_name=ANALYSIS_NAME,
    signal_ids_by_sensor=signal_ids_by_sensor,
    signal_history_by_sensor=signal_history_by_sensor,
    additional_data_by_sensor=additional_data_by_sensor,
 )
analysis_id = int(upload_result["analysis_id"])
analysis_created = bool(upload_result["analysis_created"])
if existing_analysis_id is not None and analysis_id != existing_analysis_id:
    raise AssertionError(
        f"Expected to reuse analysis id {existing_analysis_id}, but the upload returned {analysis_id}."
    )

upload_summary = pd.DataFrame(upload_result["summary"])
if not upload_summary.empty:
    upload_summary = upload_summary.sort_values(
        ["sensor_identifier", "metric", "series_key"]
    ).reset_index(drop=True)

display(upload_summary)
display(
    pd.DataFrame(
        [
            {
                "analysis_id": analysis_id,
                "analysis_created": analysis_created,
                "series_uploaded": len(upload_summary),
                "expected_series": len(unique_sensors) * len(CEIT_METRICS),
            }
        ]
    ).T.rename(columns={0: "value"})
)


,name,model_definition_id,location_id,source_type,source,description,additional_data
0,CeitCorrosionMonitoring,18,3443,ceit-json,/home/pietro.dantuono@24SEA.local/Projects/SMA...,Shared CEIT corrosion monitoring analysis acro...,"{'analysis_family': 'CEITSensor', 'shared_anal..."


,analysis,site,location,related_object,name_col1,name_col2,units_col1,units_col2,value_col1,value_col2,short_description,description,additional_data
0,0,77,3443,"{'type': 'shm.signal', 'id': 371}",timestamp,temperature,s,degC,"[1773317292.0, 1773317292.0]","[-40.84765, -40.84765]",DA87:temperature,CEIT temperature time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
1,0,77,3443,"{'type': 'shm.signal', 'id': 371}",timestamp,battery,s,V,"[1773317292.0, 1773317292.0]","[4.13714, 4.13714]",DA87:battery,CEIT battery time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
2,0,77,3443,"{'type': 'shm.signal', 'id': 371}",timestamp,tof,s,us,"[1773317292.0, 1773317292.0]","[1653.35546, 1653.35546]",DA87:tof,CEIT tof time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
3,0,77,3443,"{'type': 'shm.signal', 'id': 371}",timestamp,amplitude,s,count,"[1773317292.0, 1773317292.0]","[43.0, 43.0]",DA87:amplitude,CEIT amplitude time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
4,0,77,3443,"{'type': 'shm.signal', 'id': 371}",timestamp,meas_gain,s,gain,"[1773317292.0, 1773317292.0]","[1.39999, 1.39999]",DA87:meas_gain,CEIT meas_gain time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."


,sensor_identifier,metric,series_key,analysis_created,action,result_id,appended_points,signal_id
0,DA87,amplitude,DA87:amplitude,False,unchanged,4821,0,371
1,DA87,battery,DA87:battery,False,unchanged,4819,0,371
2,DA87,meas_gain,DA87:meas_gain,False,unchanged,4822,0,371
3,DA87,temperature,DA87:temperature,False,unchanged,4818,0,371
4,DA87,tof,DA87:tof,False,unchanged,4820,0,371
5,DA8F,amplitude,DA8F:amplitude,False,unchanged,4826,0,369
6,DA8F,battery,DA8F:battery,False,unchanged,4824,0,369
7,DA8F,meas_gain,DA8F:meas_gain,False,unchanged,4827,0,369
8,DA8F,temperature,DA8F:temperature,False,unchanged,4823,0,369
9,DA8F,tof,DA8F:tof,False,unchanged,4825,0,369


,value
analysis_id,58
analysis_created,False
series_uploaded,15
expected_series,15


## Retrieve and Verify the Persisted Results

This stage reads the stored rows back from the API and proves that the uploaded observations can be reconstructed into the normalized CEIT shape again.

**Verification performed here**

- Fetch all raw backend rows for the selected analysis id.
- Deserialize them into `ResultSeries` objects.
- Reconstruct the normalized long frame from the persisted rows.
- Verify that every normalized observation from the input JSON is present in the retrieved frame with an attached signal identifier.

*Outcome:* the notebook confirms the full round-trip from JSON to backend and back into the CEIT model layer.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["results_api.list_results(analysis=analysis_id)"] --> B["raw_results_frame"]
    C["service.load_backend_series(analysis=analysis_id)"] --> D["retrieved_series"]
    D --> E["service.frame_from_series(retrieved_series)"]
    E --> F["retrieved_frame"]
    G["measurement_frame"] --> H["input_counts"]
    F --> I["retrieved_counts"]
    F --> J["signal_lookup"]
    H --> K["round_trip_counts"]
    I --> K
    J --> K
    K --> L{"count_match.all() and signal_id.notna().all()?"}
    L -- yes --> M["round_trip_counts"]
    L -- no --> N["raise AssertionError"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;
    classDef decision fill:#FDEBD3,stroke:#C47A2C,color:#5A3412;

    class A,C,E api;
    class H,I,J,K transform;
    class B,D,F,G,M output;
    class L,N decision;
```

In [10]:
raw_results_frame = results_api.list_results(analysis=analysis_id)["data"]
display(raw_results_frame.head())
retrieved_series = service.load_backend_series(analysis=analysis_id)
display(retrieved_series)
retrieved_frame = service.frame_from_series(retrieved_series)
display(retrieved_frame.head())


,id,name_col1,name_col2,name_col3,units_col1,units_col2,units_col3,value_col1,value_col2,value_col3,analysis,site,location,short_description,description,related_object,additional_data,slug,visibility,visibility_groups
0,4821,timestamp,amplitude,None,s,count,None,"[1773317292.0, 1773317292.0]","[43.0, 43.0]",None,58,77,3443,DA87:amplitude,CEIT amplitude time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'amplitude', 'signal_id': 371, 'inp...",willow_ceit_ceitcorrosionmonitoring_58_da87amp...,usergroup,[]
1,4819,timestamp,battery,None,s,V,None,"[1773317292.0, 1773317292.0]","[4.13714, 4.13714]",None,58,77,3443,DA87:battery,CEIT battery time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'battery', 'signal_id': 371, 'input...",willow_ceit_ceitcorrosionmonitoring_58_da87bat...,usergroup,[]
2,4822,timestamp,meas_gain,None,s,gain,None,"[1773317292.0, 1773317292.0]","[1.39999, 1.39999]",None,58,77,3443,DA87:meas_gain,CEIT meas_gain time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'meas_gain', 'signal_id': 371, 'inp...",willow_ceit_ceitcorrosionmonitoring_58_da87mea...,usergroup,[]
3,4818,timestamp,temperature,None,s,degC,None,"[1773317292.0, 1773317292.0]","[-40.84765, -40.84765]",None,58,77,3443,DA87:temperature,CEIT temperature time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'temperature', 'signal_id': 371, 'i...",willow_ceit_ceitcorrosionmonitoring_58_da87tem...,usergroup,[]
4,4820,timestamp,tof,None,s,us,None,"[1773317292.0, 1773317292.0]","[1653.35546, 1653.35546]",None,58,77,3443,DA87:tof,CEIT tof time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'tof', 'signal_id': 371, 'input_fil...",willow_ceit_ceitcorrosionmonitoring_58_da87tof...,usergroup,[]


[ResultSeries(analysis_name='unknown', analysis_kind=<AnalysisKind.TIME_SERIES: 'time_series'>, result_scope=<ResultScope.LOCATION: 'location'>, short_description='DA87:amplitude', description='CEIT amplitude time series for sensor DA87.', site_id=77, location_id=3443, related_object=RelatedObject(type='shm.signal', id=371), data_additional={'metric': 'amplitude', 'signal_id': 371, 'input_file': 'MeasFile_24sea.json', 'series_key': 'DA87:amplitude', 'result_scope': 'location', 'source_field': 'Amplitude', 'analysis_kind': 'time_series', 'match_strategy': None, 'signal_history': {'id': 435, 'status': 'ok', 'signal_id': 371, 'check_date': None, 'checked_by': None, 'visibility': 'usergroup', 'description': None, 'sort_timestamp': '1970-01-01T00:00:00+00:00', 'status_approval': 'pending', 'status_priority': 0, 'is_latest_status': True, 'legacy_signal_id': None, 'sensor_record_id': 92, 'visibility_groups': [], 'sensor_serial_number': 92.0, 'activity_start_timestamp': '1970-01-01T00:00:00', 

,analysis_name,short_description,sensor_identifier,metric,signal_id,timestamp,value
0,unknown,DA87:amplitude,DA87,amplitude,371,2026-03-12T12:08:12+00:00,43.00000
1,unknown,DA87:amplitude,DA87,amplitude,371,2026-03-12T12:08:12+00:00,43.00000
2,unknown,DA87:battery,DA87,battery,371,2026-03-12T12:08:12+00:00,4.13714
3,unknown,DA87:battery,DA87,battery,371,2026-03-12T12:08:12+00:00,4.13714
4,unknown,DA87:meas_gain,DA87,meas_gain,371,2026-03-12T12:08:12+00:00,1.39999


In [11]:
raw_results_frame = results_api.list_results(analysis=analysis_id)["data"]
retrieved_series = service.load_backend_series(analysis=analysis_id)
retrieved_frame = service.frame_from_series(retrieved_series)

input_counts = (
    measurement_frame.groupby(["sensor_identifier", "metric", "timestamp", "value"]).size()
    .rename("input_count")
    .reset_index()
 )
retrieved_counts = (
    retrieved_frame.groupby(["sensor_identifier", "metric", "timestamp", "value"]).size()
    .rename("retrieved_count")
    .reset_index()
 )
signal_lookup = (
    retrieved_frame.groupby(["sensor_identifier", "metric", "timestamp", "value"])["signal_id"]
    .first()
    .reset_index()
 )

round_trip_counts = input_counts.merge(
    retrieved_counts,
    on=["sensor_identifier", "metric", "timestamp", "value"],
    how="left",
).merge(
    signal_lookup,
    on=["sensor_identifier", "metric", "timestamp", "value"],
    how="left",
)
round_trip_counts["retrieved_count"] = round_trip_counts["retrieved_count"].fillna(0).astype(int)
round_trip_counts["count_match"] = (
    round_trip_counts["retrieved_count"] >= round_trip_counts["input_count"]
 )
round_trip_counts = round_trip_counts.sort_values(
    ["sensor_identifier", "metric", "timestamp"]
).reset_index(drop=True)

display(raw_results_frame.head())
display(retrieved_frame.head())
display(round_trip_counts)

if not bool(round_trip_counts["count_match"].all()):
    raise AssertionError(
        "Some normalized CEIT observations were not recovered from the backend result rows."
    )
if bool(round_trip_counts["signal_id"].isna().any()):
    raise AssertionError(
        "At least one recovered CEIT observation is missing its linked SHM signal id."
    )


,id,name_col1,name_col2,name_col3,units_col1,units_col2,units_col3,value_col1,value_col2,value_col3,analysis,site,location,short_description,description,related_object,additional_data,slug,visibility,visibility_groups
0,4821,timestamp,amplitude,None,s,count,None,"[1773317292.0, 1773317292.0]","[43.0, 43.0]",None,58,77,3443,DA87:amplitude,CEIT amplitude time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'amplitude', 'signal_id': 371, 'inp...",willow_ceit_ceitcorrosionmonitoring_58_da87amp...,usergroup,[]
1,4819,timestamp,battery,None,s,V,None,"[1773317292.0, 1773317292.0]","[4.13714, 4.13714]",None,58,77,3443,DA87:battery,CEIT battery time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'battery', 'signal_id': 371, 'input...",willow_ceit_ceitcorrosionmonitoring_58_da87bat...,usergroup,[]
2,4822,timestamp,meas_gain,None,s,gain,None,"[1773317292.0, 1773317292.0]","[1.39999, 1.39999]",None,58,77,3443,DA87:meas_gain,CEIT meas_gain time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'meas_gain', 'signal_id': 371, 'inp...",willow_ceit_ceitcorrosionmonitoring_58_da87mea...,usergroup,[]
3,4818,timestamp,temperature,None,s,degC,None,"[1773317292.0, 1773317292.0]","[-40.84765, -40.84765]",None,58,77,3443,DA87:temperature,CEIT temperature time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'temperature', 'signal_id': 371, 'i...",willow_ceit_ceitcorrosionmonitoring_58_da87tem...,usergroup,[]
4,4820,timestamp,tof,None,s,us,None,"[1773317292.0, 1773317292.0]","[1653.35546, 1653.35546]",None,58,77,3443,DA87:tof,CEIT tof time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'tof', 'signal_id': 371, 'input_fil...",willow_ceit_ceitcorrosionmonitoring_58_da87tof...,usergroup,[]


,analysis_name,short_description,sensor_identifier,metric,signal_id,timestamp,value
0,unknown,DA87:amplitude,DA87,amplitude,371,2026-03-12T12:08:12+00:00,43.00000
1,unknown,DA87:amplitude,DA87,amplitude,371,2026-03-12T12:08:12+00:00,43.00000
2,unknown,DA87:battery,DA87,battery,371,2026-03-12T12:08:12+00:00,4.13714
3,unknown,DA87:battery,DA87,battery,371,2026-03-12T12:08:12+00:00,4.13714
4,unknown,DA87:meas_gain,DA87,meas_gain,371,2026-03-12T12:08:12+00:00,1.39999


,sensor_identifier,metric,timestamp,value,input_count,retrieved_count,signal_id,count_match
0,DA87,amplitude,2026-03-12T12:08:12+00:00,43.00000,2,2,371,True
1,DA87,battery,2026-03-12T12:08:12+00:00,4.13714,2,2,371,True
2,DA87,meas_gain,2026-03-12T12:08:12+00:00,1.39999,2,2,371,True
3,DA87,temperature,2026-03-12T12:08:12+00:00,-40.84765,2,2,371,True
4,DA87,tof,2026-03-12T12:08:12+00:00,1653.35546,2,2,371,True
5,DA8F,amplitude,2026-03-12T12:05:12+00:00,43.00000,2,2,369,True
6,DA8F,amplitude,2026-03-12T12:09:12+00:00,43.00000,2,2,369,True
7,DA8F,battery,2026-03-12T12:05:12+00:00,4.13714,2,2,369,True
8,DA8F,battery,2026-03-12T12:09:12+00:00,4.13714,2,2,369,True
9,DA8F,meas_gain,2026-03-12T12:05:12+00:00,1.39999,2,2,369,True


## Plot the Retrieved Results

The final stage renders the CEIT results from the retrieved backend rows, not from the raw input file.

**What to inspect in the plot**

- The dropdown should expose one entry per sensor identifier.
- Each sensor chart should show the five CEIT metrics as separate time-series lines.
- The plotted data should match the uploaded timestamps and values that passed the round-trip validation step.

*Outcome:* the same persisted analysis can be inspected interactively through the CEIT plotting layer exposed by the SDK.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["analysis_id"] --> B["shared_plot = service.plot_shared_analysis(analysis_id=analysis_id)"]
    B --> C["shared_plot"]
    C --> D["shared_plot.notebook"]
    D --> E["display(shared_plot.notebook)"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class B api;
    class A,C,D,E output;
```

In [12]:
shared_plot = service.plot_shared_analysis(analysis_id=analysis_id)
display(shared_plot.notebook)


In [13]:
final_summary = pd.DataFrame(
    [
        {
            "analysis_id": analysis_id,
            "analysis_created": analysis_created,
            "resolved_sensors": len(signal_ids_by_sensor),
            "persisted_series": 0 if raw_results_frame.empty else raw_results_frame["short_description"].nunique(),
            "input_rows": len(measurements),
            "normalized_input_observations": len(measurement_frame),
            "retrieved_observations": len(retrieved_frame),
            "round_trip_complete": bool(round_trip_counts["count_match"].all()),
            "plot_available": shared_plot.notebook is not None,
        }
    ]
).T.rename(columns={0: "value"})

display(final_summary)


,value
analysis_id,58
analysis_created,False
resolved_sensors,3
persisted_series,15
input_rows,8
normalized_input_observations,40
retrieved_observations,40
round_trip_complete,True
plot_available,True
